In [0]:
mport sys
import importlib

# Copy the Workspace path of your src folder and paste it here
src_path = "/Users/roy.sourav2700@gmail.com/etl_severn_trent/src"

if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Clears any wrongly cached/empty common_utils package
sys.modules.pop("com_utils", None)



In [0]:
import src.com_utils
importlib.reload(src.com_utils)

print(src.com_utils.__file__)
print(dir(src.com_utils))

In [0]:
import re
from datetime import datetime

from pyspark.sql import functions as F
from src.com_utils import apply_scd1, run_dq_checks

catalog = "severn_trent"
silver_schema = "silver"
raw_data_path = "/Volumes/severn_trent/bronze/raw_data"

# Find folders such as 01_04_2026 and 02_04_2026
snapshot_folders = [
    item.path.rstrip("/")
    for item in dbutils.fs.ls(raw_data_path)
    if item.isDir()
    and re.fullmatch(r"\d{2}_\d{2}_\d{4}", item.name.rstrip("/"))
]

if not snapshot_folders:
    raise ValueError(f"No snapshot folders found in {raw_data_path}")

# Select the most recent snapshot folder
latest_snapshot_path = max(
    snapshot_folders,
    key=lambda path: datetime.strptime(path.split("/")[-1], "%d_%m_%Y")
)

snapshot_date = latest_snapshot_path.split("/")[-1]

facts = {
    "FactShift": "shift_id",
    "FactShrinkage": "shrinkage_id",
    "FactWork": "work_id"
}

fact_rules = {
    "FactShift": {
        "key": "shift_id",
        "required_columns": [
            "shift_id", "people_id", "county_id",
            "scheduled_start", "scheduled_end"
        ],
        "date_rules": [("scheduled_start", "scheduled_end")],
        "foreign_keys": [
            {
                "child_column": "people_id",
                "reference_table": "severn_trent.silver.DimPeople",
                "reference_column": "people_id"
            },
            {
                "child_column": "county_id",
                "reference_table": "severn_trent.silver.DimCounty",
                "reference_column": "county_id"
            }
        ]
    },
    "FactShrinkage": {
        "key": "shrinkage_id",
        "required_columns": [
            "shrinkage_id", "people_id", "shrinkage_type_id",
            "start_datetime", "end_datetime"
        ],
        "date_rules": [("start_datetime", "end_datetime")],
        "foreign_keys": [
            {
                "child_column": "people_id",
                "reference_table": "severn_trent.silver.DimPeople",
                "reference_column": "people_id"
            },
            {
                "child_column": "shrinkage_type_id",
                "reference_table": "severn_trent.silver.DimShrinkageType",
                "reference_column": "shrinkage_type_id"
            }
        ]
    },
    "FactWork": {
        "key": "work_id",
        "required_columns": [
            "work_id", "people_id", "county_id", "status",
            "scheduled_start", "scheduled_end"
        ],
        "date_rules": [("scheduled_start", "scheduled_end")],
        "foreign_keys": [
            {
                "child_column": "people_id",
                "reference_table": "severn_trent.silver.DimPeople",
                "reference_column": "people_id"
            },
            {
                "child_column": "county_id",
                "reference_table": "severn_trent.silver.DimCounty",
                "reference_column": "county_id"
            }
        ]
    }
}


for fact_name, business_key in facts.items():
    source_path = f"{latest_snapshot_path}/{fact_name}_{snapshot_date}.csv"

    # SCD1 table: contains only the latest record per business key.
    target_table = f"{catalog}.{silver_schema}.{fact_name}"

    print(f"Processing {source_path}")
    print(f"Writing to {target_table}")

    delta_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_path)
        .dropDuplicates()
        .withColumn("last_update_ts", F.current_timestamp())
    )
    
    # DQ rules for the current fact table
    rules = fact_rules[fact_name]

    # Run DQ checks and send failed records to dq_check schema
    valid_df, dq_status = run_dq_checks(
        spark=spark,
        source_df=delta_df,
        source_name=fact_name,
        primary_keys=[rules["key"]],
        required_columns=rules["required_columns"],
        date_order_rules=rules["date_rules"],
        foreign_keys=rules["foreign_keys"]
    )

    # Do not load invalid rows into Silver
    if valid_df.limit(1).count() == 0:
        print(f"No valid rows available for {fact_name}; Silver load skipped.")
        continue

    apply_scd1(
        spark=spark,
        source_df=delta_df,             # Must be the DataFrame, not raw_data_path
        target_table=target_table,
        join_keys=[business_key],       # Must be a list
        watermark_column="last_update_ts",
        full_refresh=False
    )

print(f"Facts processed for snapshot {snapshot_date}.")